# World Serializer Exploration

This notebook exercises `llmr.bridge.world_reader` only. It loads the same PR2 apartment world fixture used by `runner.ipynb`, then inspects how `serialize_world_from_symbol_graph()` turns KRROOD `SymbolGraph` state into LLM-readable context.

No LLM calls are made here.

## Stage 0 - Load World

`symbol_type = Body` scopes the serializer to physical bodies instead of every `Symbol` instance.

In [ ]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body
from krrood.symbol_graph.symbol_graph import SymbolGraph

from llmr.bridge.world_reader import (
    WorldSerializationOptions,
    body_bounding_box,
    body_display_name,
    body_xyz,
    serialize_world_from_symbol_graph,
)

world, pr2, context = load_pr2_apartment_world()
graph = SymbolGraph()
groundable_type = Body

bodies = list(graph.get_instances_of_type(groundable_type))
# print(f"World loaded: {world}")
# print(f"Robot      : {pr2}")
# # print(f"Bodies in SymbolGraph for {groundable_type.__name__}: {len(bodies)}")
# for body in bodies[:30]:
#     print(f"  - {body_display_name(body)}")
# if len(bodies) > 30:
#     print(f"  ... {len(bodies) - 30} more")

## Stage 1 - Default Structured Context

Default output is Structured Markdown optimized for smaller LLMs: summary, grounding instructions, scene table, semantic types, spatial context, and SymbolGraph relations.

In [ ]:
world_context = render_world_context(symbol_type)
print(world_context)

## Stage 2 - Prompt Size Controls

Use `WorldContextConfig` to limit objects/relations or suppress geometry. These options are useful when testing small local models with tight context windows.

In [ ]:
compact_options = WorldContextConfig(
    max_objects=117,
    max_relations=10,
    include_geometry=True,
    include_parent_context=False,
    include_relations=True,
    include_structural=False,
    exclude_robot_structures=True
)

compact_context = render_world_context(
    symbol_type,
    options=compact_options,
)
print(compact_context)

## Stage 3 - Structural Robot Filtering

By default, robot-owned kinematic bodies are excluded through semantic robot annotations when possible. Name suffix filtering is kept only as a fallback. Toggle `include_structural=True` to inspect everything.

In [ ]:
filtered = render_world_context(
    symbol_type,
    options=WorldContextConfig(max_objects=25),
)

with_structural = render_world_context(
    symbol_type,
    options=WorldContextConfig(max_objects=25, include_structural=True),
)

print("=== Default: structural bodies filtered ===")
print(filtered)
print("\n\n=== include_structural=True ===")
print(with_structural)

## Stage 4 - Custom Structural Filter

A caller can provide a domain-specific `structural_body_filter`. The filter below hides any body whose display name contains `camera`, independent of semantic annotations or suffixes.

In [ ]:
def hide_camera_named_bodies(body):
    return "camera" in symbol_display_name(body).lower()

custom_context = render_world_context(
    symbol_type,
    options=WorldContextConfig(
        max_objects=30,
        structural_body_filter=hide_camera_named_bodies,
    ),
)
print(custom_context)

## Stage 5 - Extra Context Section

`extra_context` is appended under `## Extra Context`. This is useful for temporary runtime hints that are not part of `SymbolGraph`.

In [ ]:
extra_context = render_world_context(
    symbol_type,
    extra_context="Prefer objects on the table when the instruction is ambiguous.",
    options=WorldContextConfig(max_objects=15),
)
print(extra_context)

## Stage 6 - Inspect Raw Helper Values

The serializer uses duck-typed helpers for names, positions, and bounding boxes. This cell prints those raw values for a few bodies so serializer rows can be compared against source data.

In [ ]:
for body in bodies[:20]:
    name = symbol_display_name(body)
    xyz = symbol_xyz(body)
    bbox = symbol_bounding_box(body)
    parent_conn = getattr(body, "parent_connection", None)
    parent = getattr(parent_conn, "parent", None) if parent_conn else None
    parent_name = symbol_display_name(parent) if parent is not None else None
    print(f"{name:40s} parent={parent_name!r:25} xyz={xyz!r:35} bbox={bbox!r}")

## Stage 7 - Show Semantic Annotation Sources

This mirrors the serializer's annotation scan: any SymbolGraph instance with a `.candidates` attribute can contribute semantic types to the scene-object table.

In [ ]:
for wrapped in graph.wrapped_instances:
    inst = wrapped.instance
    if inst is None or not hasattr(inst, "bodies"):
        continue
    try:
        body_names = [symbol_display_name(body) for body in inst.candidates]
    except Exception as exc:
        body_names = [f"<unavailable: {exc}>"]
    if body_names:
        print(f"{type(inst).__name__:40s} -> {body_names}")